In [1]:
"""
Build assumption_master sheet: spouse_factors
-----
Extracts Spouse-specific rating factors. Consumed by Spouse GLA/LSDB
pricing rows.

Schema: factor_type | cost_basis | gender | value | notes
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "spouse_factors"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")

Building: spouse_factors


In [2]:
# Preview rows 355-370 cols B-D
print("Spouses Cover factors block (rows 355-370, cols B-D):")
print("-" * 90)
for row in range(355, 371):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row:3d}: B={b!r:70s} C={c!r:10s} D={d!r}")

Spouses Cover factors block (rows 355-370, cols B-D):
------------------------------------------------------------------------------------------
  Row 355: B=None                                                                   C=None       D=None
  Row 356: B="Spouse's Cover - Basic Spouses Factors: (based on member's gender)"   C=None       D=None
  Row 357: B='M'                                                                    C=-0.55      D=None
  Row 358: B='F'                                                                    C=1.7        D=None
  Row 359: B=None                                                                   C=None       D=None
  Row 360: B="Spouse's Cover - Basic Spouses Factors: (based on member's gender)"   C=None       D=None
  Row 361: B='M'                                                                    C=None       D=None
  Row 362: B='F'                                                                    C=None       D=None
  Row 363: B=None      

In [3]:
all_rows: list[dict] = []

def add_row(factor_type, cost_basis, gender, value, source_row, notes=""):
    all_rows.append({
        "factor_type": factor_type,
        "cost_basis": cost_basis,
        "gender": gender,
        "value": value,
        "source_row": source_row,
        "notes": notes,
    })

# =============================================================================
# Basic Spouses Factors (BASIC cost_basis) — rows 357-358, col C
# Consumed by Spouse GLA row 22 and Spouse LSDB row 22 via SP_Factors named range
# =============================================================================
add_row(
    "basic_spouses_factor", "basic", "M",
    ws.cell(row=357, column=3).value, 357,
    "Basic spouse factor when member is male (SP_Factors)",
)
add_row(
    "basic_spouses_factor", "basic", "F",
    ws.cell(row=358, column=3).value, 358,
    "Basic spouse factor when member is female (SP_Factors)",
)

# =============================================================================
# AIDS Spouses Factors — rows 361-362, col C
# BLANK in source A2026_07 (placeholder for future AIDS variant).
# Preserved as null per rule 5.5. Consumed via SP_Factors_Aids named range;
# pricing engine INDEX call will return null → engine must handle downstream.
# =============================================================================
add_row(
    "basic_spouses_factor", "aids", "M",
    ws.cell(row=361, column=3).value, 361,
    "AIDS spouse factor when member is male (SP_Factors_Aids). "
    "BLANK in source A2026_07 — placeholder for future AIDS variant.",
)
add_row(
    "basic_spouses_factor", "aids", "F",
    ws.cell(row=362, column=3).value, 362,
    "AIDS spouse factor when member is female (SP_Factors_Aids). "
    "BLANK in source A2026_07 — placeholder for future AIDS variant.",
)

# =============================================================================
# DB Definition Factors — vertical layout: label in row N col B, value in row N+1 col B
# Consumed via Spouses_Basic_DB_Definition_Factor named range.
# =============================================================================
db_basic_val = ws.cell(row=366, column=2).value  # B366
add_row(
    "db_definition_factor", "basic", "all",
    db_basic_val, 366,
    "Spouses Basic DB Definition Factor (Spouses_Basic_DB_Definition_Factor). "
    "Vertical layout: label at B365, value at B366.",
)

db_aids_val = ws.cell(row=369, column=2).value  # B369
add_row(
    "db_definition_factor", "aids", "all",
    db_aids_val, 369,
    "Spouses AIDS DB Definition Factor. Vertical layout: label at B368, value at B369.",
)

print(f"Total rows harvested: {len(all_rows)}")
print(f"\nAll rows preview:")
for r in all_rows:
    print(f"  {r['factor_type']:22s} | {r['cost_basis']:6s} | {r['gender']:3s} | "
          f"value={r['value']!r} ({type(r['value']).__name__})")

Total rows harvested: 6

All rows preview:
  basic_spouses_factor   | basic  | M   | value=-0.55 (float)
  basic_spouses_factor   | basic  | F   | value=1.7 (float)
  basic_spouses_factor   | aids   | M   | value=None (NoneType)
  basic_spouses_factor   | aids   | F   | value=None (NoneType)
  db_definition_factor   | basic  | all | value=-0.9 (float)
  db_definition_factor   | aids   | all | value=-0.9 (float)


In [4]:
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

for r in all_rows:
    v = r["value"]
    r["param_type"] = classify(v)
    r["value_str"] = "" if v is None else str(v)

df = pl.DataFrame(all_rows).select([
    "factor_type",
    "cost_basis",
    "gender",
    pl.col("value_str").alias("value"),
    "param_type",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
key_cols = ["factor_type", "cost_basis", "gender"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

# Row count
assert df.height == 6, f"Expected 6 rows, got {df.height}"

# Expected factor_types present
expected_types = {"basic_spouses_factor", "db_definition_factor"}
actual_types = set(df["factor_type"].unique().to_list())
assert actual_types == expected_types, (
    f"Type mismatch: expected {expected_types}, got {actual_types}"
)

# =============================================================================
# Frozen value expectations
# =============================================================================
def get_val(factor_type, cost_basis, gender):
    row = df.filter(
        (pl.col("factor_type") == factor_type)
        & (pl.col("cost_basis") == cost_basis)
        & (pl.col("gender") == gender)
    )
    assert row.height == 1, f"Missing: {factor_type}/{cost_basis}/{gender}"
    return row["value"][0], row["param_type"][0]

# Basic spouses factor - basic - M/F
val_m, typ_m = get_val("basic_spouses_factor", "basic", "M")
assert typ_m == "float"
assert abs(float(val_m) - (-0.55)) < 1e-9, f"basic M: got {val_m}, expected -0.55"

val_f, typ_f = get_val("basic_spouses_factor", "basic", "F")
assert typ_f == "float"
assert abs(float(val_f) - 1.7) < 1e-9, f"basic F: got {val_f}, expected 1.7"

# AIDS variant - must be null
for gender in ("M", "F"):
    val, typ = get_val("basic_spouses_factor", "aids", gender)
    assert typ == "null", (
        f"AIDS {gender}: expected null, got type={typ} value={val!r}"
    )

# DB definition factor - basic and aids both -0.9
val_db_basic, typ_db_basic = get_val("db_definition_factor", "basic", "all")
assert typ_db_basic == "float"
assert abs(float(val_db_basic) - (-0.9)) < 1e-9

val_db_aids, typ_db_aids = get_val("db_definition_factor", "aids", "all")
assert typ_db_aids == "float"
assert abs(float(val_db_aids) - (-0.9)) < 1e-9

print(f"✓ {df.height} rows validated")
print(f"✓ Basic factors: M=-0.55, F=1.7 (both float)")
print(f"✓ AIDS factors: both null (rule 5.5 preserved)")
print(f"✓ DB definition factors: basic=-0.9, aids=-0.9")

print(f"\nType distribution:")
print(df.group_by("param_type").len().sort("param_type"))

print(f"\nFinal DataFrame:")
print(df)

✓ 6 rows validated
✓ Basic factors: M=-0.55, F=1.7 (both float)
✓ AIDS factors: both null (rule 5.5 preserved)
✓ DB definition factors: basic=-0.9, aids=-0.9

Type distribution:
shape: (2, 2)
┌────────────┬─────┐
│ param_type ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ float      ┆ 4   │
│ null       ┆ 2   │
└────────────┴─────┘

Final DataFrame:
shape: (6, 7)
┌─────────────────────┬────────────┬────────┬───────┬────────────┬────────────┬────────────────────┐
│ factor_type         ┆ cost_basis ┆ gender ┆ value ┆ param_type ┆ source_row ┆ notes              │
│ ---                 ┆ ---        ┆ ---    ┆ ---   ┆ ---        ┆ ---        ┆ ---                │
│ str                 ┆ str        ┆ str    ┆ str   ┆ str        ┆ i64        ┆ str                │
╞═════════════════════╪════════════╪════════╪═══════╪════════════╪════════════╪════════════════════╡
│ basic_spouses_facto ┆ basic      ┆ M      ┆ -0.55 ┆ float      ┆ 357        ┆ Basic spouse       │
│ 

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote spouse_factors.parquet
  Size:    3.30 KB
  SHA-256: 6c71aa35fe4fe538...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "Spouse rating factors. 6 rows: basic_spouses_factor (M/F × basic/aids, "
        "AIDS variant blank in source — preserved null per rule 5.5) and "
        "db_definition_factor (basic/aids, both -0.9). DB definition factors "
        "have VERTICAL layout in source (label row N col B, value row N+1 col B)."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated

BUILD COMPLETE: spouse_factors
